#  AegisHealth KBQA — Benchmark N1: Baseline (LLM thuần)

Notebook này đo **fabrication rate** của LLM khi trả lời câu hỏi y khoa **không có RAG/KG**
(closed-book) — bậc 1 trong ablation 3 bậc:

```
N1 (LLM thuần)  →  N2 (+RAG vector / LightRAG naive)  →  N3 (+KG Hybrid Cypher+LightRAG)
```

## Kiến trúc
```
Câu hỏi (golden_test_v2.json)
        │
        
AsyncOpenAI  (qwen2.5:3b qua Ollama local — closed-book, KHÔNG run_pipeline)
        │
        
raw_baseline.jsonl  ──  score_golden.score_run  ──  results_baseline.json + report_baseline.md
```

##  Checklist trước khi Run All
- [ ] **Accelerator** → GPU T4 x1 (tùy chọn — qwen2.5:3b chạy được cả CPU, GPU nhanh hơn)
- [ ] **Internet** → bật (clone repo qua git + `ollama pull`)
- [ ] **Add Data** → attach Kaggle Dataset `aegishealth-benchmark`
      (chứa `golden_test_v2.json` + `kg_entities.txt`, ~2MB)
- [ ] Branch `refactor/review-architecture` đã được **push** lên GitHub
      (chứa `ai_engine/eval/score_golden.py` + `data/benchmark/golden_test_v2.json` ở repo
      chỉ dùng làm fallback nếu dataset chưa attach)

##  Ước tính: ~15-25 phút (100 câu × qwen2.5:3b)

> Notebook này KHÔNG cần Neo4j / Qdrant / bge-m3 — chỉ Ollama + qwen2.5:3b.


In [ ]:
# --- Kiểm tra môi trường ---
import platform
import subprocess
import sys

print("=" * 60)
print(" Environment Check")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        text=True,
    ).strip()
    print(f"GPU: {gpu_info}")
except Exception:
    print(" No GPU detected — qwen2.5:3b vẫn chạy được trên CPU, chỉ chậm hơn")

with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith("MemTotal"):
            ram_gb = int(line.split()[1]) / 1024 / 1024
            print(f"RAM: {ram_gb:.1f} GB")
            break

print("\n Environment check done")


In [ ]:
# --- Cấu hình ---
MODEL_NAME = "qwen2.5:3b"

OLLAMA_PORT = 11434
INFERENCE_TIMEOUT = 90  # giây / câu

REPO_URL = "https://github.com/NgBaoAnn/kbqa.git"
REPO_BRANCH = "refactor/review-architecture"  # ← branch chứa score_golden.py, phải push trước
REPO_DIR = "/kaggle/working/kbqa"

DATASET_DIR = "/kaggle/input/aegishealth-benchmark"
OUTPUT_DIR = "/kaggle/working"

print(" Config:")
print(f"  Model:       {MODEL_NAME}")
print(f"  Repo branch: {REPO_BRANCH}")


In [ ]:
# --- Clone repo (chỉ cần score_golden.py — không pip install) ---
import os
import subprocess
import sys

if os.path.exists(REPO_DIR):
    print(f" Repo đã tồn tại, đang cập nhật branch {REPO_BRANCH}...")
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
else:
    print(f" Cloning repo {REPO_URL} (branch={REPO_BRANCH})...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR],
        check=True,
    )

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ai_engine.eval.score_golden import score_run

print(" score_golden imported OK")


In [ ]:
# --- Tìm golden_test_v2.json + kg_entities.txt ---
import json
from pathlib import Path


def find_file(name: str) -> Path:
    candidates = [
        Path(DATASET_DIR) / name,
        Path(REPO_DIR) / "data" / "benchmark" / name,
    ]
    for c in candidates:
        if c.exists():
            return c
    # Fallback: Kaggle có thể mount dataset ở đường dẫn lồng nhau
    # (vd. /kaggle/input/datasets/<owner>/<slug>/...) — quét đệ quy /kaggle/input.
    for c in Path("/kaggle/input").rglob(name):
        return c
    raise FileNotFoundError(
        f"{name} không tìm thấy trong {DATASET_DIR}, {REPO_DIR}/data/benchmark, "
        f"hoặc bất kỳ đâu dưới /kaggle/input — kiểm tra Kaggle Dataset 'aegishealth-benchmark' đã attach chưa."
    )


GOLDEN_PATH = find_file("golden_test_v2.json")
KG_PATH = find_file("kg_entities.txt")

with open(GOLDEN_PATH, encoding="utf-8") as f:
    golden = json.load(f)

print(f" Golden:      {GOLDEN_PATH}  ({len(golden)} câu)")
print(f" KG entities: {KG_PATH}")


In [ ]:
# --- Cài và khởi động Ollama, pull model ---
import os
import subprocess
import time

import requests

OLLAMA_URL = f"http://localhost:{OLLAMA_PORT}"
OLLAMA_MODELS_DIR = "/kaggle/working/ollama_models"


def find_ollama():
    for p in ("/usr/local/bin/ollama", "/usr/bin/ollama", "/bin/ollama"):
        if os.path.exists(p):
            return p
    try:
        return subprocess.check_output(["which", "ollama"], text=True).strip()
    except Exception:
        return None


ollama_bin = find_ollama()

if ollama_bin:
    print(f" Ollama đã có sẵn tại: {ollama_bin}")
else:
    print("  Đang cài Ollama...")
    install_result = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True,
    )
    print(install_result.stdout[-1000:])
    ollama_bin = find_ollama()
    if not ollama_bin:
        raise RuntimeError(f"Ollama install failed (code {install_result.returncode})")
    print(f" Ollama đã cài thành công tại: {ollama_bin}")

os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
print(" Khởi động Ollama server...")
ollama_env = os.environ.copy()
ollama_env["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
ollama_env["OLLAMA_KEEP_ALIVE"] = "60m"

ollama_log = "/kaggle/working/ollama.log"
with open(ollama_log, "w") as log:
    ollama_proc = subprocess.Popen([ollama_bin, "serve"], env=ollama_env, stdout=log, stderr=log)

print(" Chờ Ollama server...")
for i in range(40):
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        if r.status_code == 200:
            print(f" Ollama server ready (sau {i + 1}s)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server timeout sau 40s")

print(f"  Pulling model: {MODEL_NAME} (có thể mất vài phút)")
t0 = time.time()
result = subprocess.run([ollama_bin, "pull", MODEL_NAME], text=True)
if result.returncode != 0:
    raise RuntimeError(f"ollama pull {MODEL_NAME} failed")
print(f" Model '{MODEL_NAME}' sẵn sàng ({time.time() - t0:.0f}s)")


In [ ]:
# --- AsyncOpenAI client + prompt closed-book ---
import asyncio
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)

from openai import AsyncOpenAI

client = AsyncOpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# Closed-book: KHÔNG có ngữ cảnh KG/RAG. Yêu cầu cùng định dạng (in đậm + gạch
# đầu dòng) như cypher_answer_synthesizer để score_golden._extract_structured
# trích mention được công bằng giữa 3 notebook.
SYSTEM_PROMPT = (
    "Bạn là trợ lý y khoa, trả lời câu hỏi y tế bằng tiếng Việt dựa HOÀN TOÀN vào kiến thức "
    "y khoa bạn đã được huấn luyện (không có quyền truy cập cơ sở dữ liệu hay Internet).\n\n"
    "Định dạng câu trả lời:\n"
    "- In đậm (**...**) tên bệnh, tên thuốc, triệu chứng, thực phẩm khi được nhắc đến.\n"
    "- Nếu câu hỏi yêu cầu liệt kê (nhiều bệnh / thuốc / triệu chứng / thực phẩm), trình bày "
    "mỗi mục trên một dòng, bắt đầu bằng dấu gạch đầu dòng (-).\n"
    "- Trả lời ngắn gọn (3-15 dòng), không thêm lời mở đầu/kết luận dài dòng, không thêm câu "
    "miễn trừ trách nhiệm.\n"
    "- Trả lời 100% bằng tiếng Việt."
)


async def ask_baseline(question: str, timeout: float = INFERENCE_TIMEOUT) -> tuple[str, str | None]:
    """Gọi LLM closed-book. Trả về (answer_text, error_code)."""
    try:
        resp = await asyncio.wait_for(
            client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question},
                ],
                temperature=0.3,
            ),
            timeout=timeout,
        )
        return resp.choices[0].message.content or "", None
    except asyncio.TimeoutError:
        return "", "TIMEOUT"
    except Exception as e:
        print(f"   LLM call failed: {e}")
        return "", "MODEL_UNAVAILABLE"


print(" ask_baseline() ready")


In [ ]:
# --- Smoke test — thử 1 câu hỏi ---
test_q = golden[0]["question"]
print(f" Smoke test: {test_q}")

answer, err = await ask_baseline(test_q)
print(f"error_code: {err}")
print(f"answer:\n{answer}")


In [ ]:
# --- Chạy inference cho toàn bộ golden set ---
import json
import time

RAW_PATH = os.path.join(OUTPUT_DIR, "raw_baseline.jsonl")

t_start = time.time()
with open(RAW_PATH, "w", encoding="utf-8") as f:
    for i, item in enumerate(golden):
        q = item["question"]
        t0 = time.time()
        answer, err = await ask_baseline(q)
        elapsed_ms = round((time.time() - t0) * 1000, 1)

        row = {
            **item,
            "system_answer": answer,
            "engine": "llm_baseline",
            "query_mode": None,
            "error_code": err,
            "elapsed_ms": elapsed_ms,
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

        print(f"[{i + 1}/{len(golden)}] {elapsed_ms:7.0f}ms err={err!s:5} {q[:60]}")

print(f"\n Done in {time.time() - t_start:.0f}s → {RAW_PATH}")


In [ ]:
# --- Chấm điểm bằng score_golden ---
report = score_run(RAW_PATH, GOLDEN_PATH, KG_PATH, use_bertscore=False)

RESULTS_PATH = os.path.join(OUTPUT_DIR, "results_baseline.json")
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

s = report["summary"]
print(f"{'=' * 55}")
print("  SCORE REPORT — N1 baseline (LLM thuần)")
print(f"{'=' * 55}")
print(f"  Total:        {s['total']}  (errors: {s['error_count']}, timeouts: {s['timeout_count']})")
print(f"  Precision:    {s['precision']:.3f}")
print(f"  Recall:       {s['recall']:.3f}")
print(f"  F1:           {s['f1']:.3f}")
print(f"  Fabrication:  {s['fabrication_rate']:.3f}  ")
print(f"  Off-answer:   {s['off_answer_rate']:.3f}")
print(f"\n Results saved → {RESULTS_PATH}")


In [ ]:
# --- Sinh report_baseline.md ---
def fmt_row(name, m):
    if not m:
        return f"| {name} | - | - | - | - | - | - | - |"
    return (
        f"| {name} | {m['n']} | {m['precision']:.3f} | {m['recall']:.3f} | "
        f"{m['f1']:.3f} | {m['fabrication_rate']:.3f} | {m['em']:.3f} | {m['hits1']:.3f} |"
    )


lines = []
lines.append(f"# Báo cáo Benchmark — N1: Baseline (LLM thuần, {MODEL_NAME})\n")
lines.append(
    f"**Tổng số câu:** {s['total']}  •  **Lỗi:** {s['error_count']}  •  "
    f"**Timeout:** {s['timeout_count']}\n"
)
lines.append("## Tổng quan\n")
lines.append(f"- Precision: **{s['precision']:.3f}**")
lines.append(f"- Recall: **{s['recall']:.3f}**")
lines.append(f"- F1: **{s['f1']:.3f}**")
lines.append(f"- Fabrication rate: **{s['fabrication_rate']:.3f}** ")
lines.append(f"- Off-answer rate: **{s['off_answer_rate']:.3f}**\n")

lines.append("## Theo regime\n")
lines.append("| regime | n | P | R | F1 | Fab | EM | Hits@1 |")
lines.append("|---|---|---|---|---|---|---|---|")
for regime, m in report["by_regime"].items():
    lines.append(fmt_row(regime, m))
lines.append("")

lines.append("## Theo complexity (hop)\n")
lines.append("| complexity | n | P | R | F1 | Fab | EM | Hits@1 |")
lines.append("|---|---|---|---|---|---|---|---|")
for hop, m in report["by_complexity"].items():
    lines.append(fmt_row(hop, m))
lines.append("")

lines.append("## Theo question_type\n")
lines.append("| question_type | n | P | R | F1 | Fab | EM | Hits@1 |")
lines.append("|---|---|---|---|---|---|---|---|")
for qt, m in report["by_type"].items():
    lines.append(fmt_row(qt, m))
lines.append("")

REPORT_PATH = os.path.join(OUTPUT_DIR, "report_baseline.md")
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f" Report saved → {REPORT_PATH}")

from IPython.display import Markdown, display

display(Markdown("\n".join(lines)))


In [ ]:
# --- Dọn dẹp ---
try:
    ollama_proc.terminate()
    print(" Ollama stopped")
except Exception as e:
    print(f" {e}")

print("\n Output files:")
for fn in ("raw_baseline.jsonl", "results_baseline.json", "report_baseline.md"):
    p = os.path.join(OUTPUT_DIR, fn)
    if os.path.exists(p):
        print(f"  {p} ({os.path.getsize(p):,} bytes)")
